# **Medical Agent:**
**LangChain Agent with Fine Tuning LLM with QLORA + IFT**

In [1]:
# Install packages by using requirements.txt
%pip install -r "/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap13/requirements.txt"

In [2]:
# Imports (1)
import torch
import accelerate
import peft
import bitsandbytes
import transformers
import trl
import datasets
import langchain

In [3]:
# Imports (2)
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import pipeline, TrainingArguments
from peft import AutoPeftModelForCausalLM, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
import warnings
warnings.filterwarnings('ignore')

## 1º - Loading the dataset

In [4]:
# Loading the dataset
medical_dataset = load_dataset('nlpie/Llama2-MedTuned-Instructions')

In [5]:
print(medical_dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source'],
        num_rows: 200252
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source'],
        num_rows: 70066
    })
})


In [6]:
# Loading training dataset
tranining_medical_dataset = medical_dataset['train'].select(indices = range(50000))
tranining_medical_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source'],
    num_rows: 50000
})

In [7]:
# Loading test dataset
test_medical_dataset = medical_dataset['train'].select(indices = range(50000, 52000))
test_medical_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source'],
    num_rows: 2000
})

In [8]:
# See dataset examples
for i in range(3):
  data = medical_dataset['train'][i]
  print(f"Data point {i + 1}:")
  print("Instruction:", data["instruction"])
  print("Input:", data["input"])
  print("Output:", data["output"])
  print("\n---------------------\n")

Data point 1:
Instruction: In your role as a medical professional, address the user's medical questions and concerns.
Input: My relative suffering from secondary lever cancer ( 4th stage as per Allopathic doctor) and primary is in rectum. He is continuously with 103 to 104 degree F fever. Allpathic doctor suggested chemo only after fever subsidises. Is treatment possible at Lavanya & what is the time scale of recover.
Output: Hi, dairy have gone through your question. I can understand your concern. He has rectal cancer with liver metastasis. It is stage 4 cancer. Surgery is not possible at this stage. Only treatment options are chemotherapy and radiotherapy according to type of cancer. Inspite of all treatment prognosis is poor. Life expectancy is not good. Consult your doctor and plan accordingly. Hope I have answered your question, if you have any doubts then contact me at bit.ly/ Chat Doctor. Thanks for using Chat Doctor. Wish you a very good health.

---------------------

Data poi

## 2º - Automate Prompt creation for Model Training

In [9]:
# Defining a function which receives a dic sample
def medical_prompt_creation(sample):

  # Defining a pre-prompt string as the first prompt part
  pre_prompt = """[INST]<<SYS>> {instruction}\n"""

  # Concatenate pre_prompt with remains strings for full prompt
  prompt = pre_prompt + "{input}" + "[/INST]" + "\n{output}"

  # Extract instruction values from dataset
  example_instruction = sample["instruction"]

  # Extract input values from dataset
  example_input = sample["input"]

  # Extract output values from dataset
  example_output = sample["output"]

  # Uses PromptTemplate from LangChain to create a template instance
  prompt_template = PromptTemplate(
      input_variables = ["instruction", "input", "output"],
      template = prompt
  )

  # Use format to create values from vars
  prompt_value_example = prompt_template.format(
      instruction = example_instruction,
      input = example_input,
      output = example_output
  )

  # Return the prompt
  return prompt_value_example

In [10]:
# Testing function (training example)
prompt = medical_prompt_creation(tranining_medical_dataset[0])
print(prompt)

[INST]<<SYS>> In your role as a medical professional, address the user's medical questions and concerns.
My relative suffering from secondary lever cancer ( 4th stage as per Allopathic doctor) and primary is in rectum. He is continuously with 103 to 104 degree F fever. Allpathic doctor suggested chemo only after fever subsidises. Is treatment possible at Lavanya & what is the time scale of recover.[/INST]
Hi, dairy have gone through your question. I can understand your concern. He has rectal cancer with liver metastasis. It is stage 4 cancer. Surgery is not possible at this stage. Only treatment options are chemotherapy and radiotherapy according to type of cancer. Inspite of all treatment prognosis is poor. Life expectancy is not good. Consult your doctor and plan accordingly. Hope I have answered your question, if you have any doubts then contact me at bit.ly/ Chat Doctor. Thanks for using Chat Doctor. Wish you a very good health.


In [11]:
# Testing function (test example)
prompt = medical_prompt_creation(test_medical_dataset[0])
print(prompt)

[INST]<<SYS>> Given your background as a doctor, please provide your insight in addressing the medical questions based on the patient's account.
Analyze the question and answer with the best option.
###Question: All are TRUE about Mammography, EXCEPT:
###Options:
A. Low energy X rays are used
B. It is a screening and diagnostic tool
C. High positive predictive value
D. Specificity of mammography is 95%
[/INST]
###Rationale: Mammography has a relatively low positive predictive value and result in high percentage of biopsy results negative for malignant growth. Mammography is widely used as a screening test for early identification of breast cancer. It is the most effective method for the detection of calcifying early stage breast cancer and has a high sensitivity for the detection of suspicious masses in fatty and fibroglandular breast tissue. Low energy X rays used in mammography produces high contrast images that differentiate normal breast tissue from possible abnormalities. It has a

## 3º - Quantization Process

In [12]:
# Base model with 4 bits only
use_4bit = True

In [13]:
# Defining dtype for base model
bnb_4bit_compute_dtype = "float16"

In [14]:
# Defining quantization type
bnb_4bit_quant_type = "nf4"

In [15]:
# Deactivate duple quantization
use_nested_quant = False

In [16]:
# Defining dtype for pytorch computation
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

In [17]:
# Defining quantization configuration
bnb_config = BitsAndBytesConfig(
              load_in_4bit = use_4bit,
              bnb_4bit_use_double_quant = use_nested_quant,
              bnb_4bit_quant_type = bnb_4bit_quant_type,
              bnb_4bit_compute_dtype = bnb_4bit_compute_dtype
             )

In [18]:
# See if GPU supports bfloat16
if compute_dtype == torch.float16 and use_4bit:
  major, _ = torch.cuda.get_device_capability()
  if major >= 8:
    print("=" * 30)
    print("Your GPU supports bfloat16!!")
    print("=" * 30)

Your GPU supports bfloat16!!


## 4º - Loading LLM and Tokenizer

In [19]:
# LLM
llm_name = "NousResearch/Llama-2-7b-chat-hf"

In [20]:
# Loading Tokenizer
tokenizer = AutoTokenizer.from_pretrained(llm_name)

In [21]:
# Use Tokenizer EOS for padding
tokenizer.pad_token = tokenizer.eos_token

In [22]:
# Right padding
tokenizer.padding_side = "right"

In [23]:
# Loading LLM with Quantization Config
model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    quantization_config = bnb_config,
    device_map = "auto",
    use_cache = False
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 5º - LORA Configuration

In [24]:
# Lora parameters
lora_config = LoraConfig(
      r = 8,
      lora_alpha = 16,
      lora_dropout = 0.05,
      bias = "none",
      task_type = "CAUSAL_LM"
    )

In [25]:
# Prepare model for PEFT
model = prepare_model_for_kbit_training(model)

In [26]:
# Quanto model + PEFT Config
model = get_peft_model(model, lora_config)

## 6º - Fine Tunning LLM

In [27]:
# Output model
output_model = "drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap13/output"

In [28]:
# Training Arguments for Trainer
training_arguments = TrainingArguments(
      output_dir = output_model,
      per_device_train_batch_size = 2,
      gradient_accumulation_steps = 4,
      optim = "paged_adamw_32bit",
      learning_rate = 2e-4,
      lr_scheduler_type = "cosine",
      save_strategy = "epoch",
      logging_steps = 10,
      num_train_epochs = 10,
      max_steps = 100,
      fp16 = True,
      report_to = "none"
    )

In [29]:
# SFTConfig
sft_config = SFTConfig(
    packing = True,
    dataset_text_field = "instruction"
)

In [30]:
# Trainer creation
medical_trainer = SFTTrainer(
    model = model,
    train_dataset = tranining_medical_dataset,
    eval_dataset = test_medical_dataset,
    args = training_arguments,
    formatting_func = medical_prompt_creation,
    peft_config = lora_config
    # max_seq_length = 512,
    # tokenizer = tokenizer,
    # packing = True
)

Applying formatting function to train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [31]:
# Fine tuning the model
medical_trainer.train()

Step,Training Loss
10,2.583800
20,2.319000
30,1.886500
40,1.727700
50,1.690400
60,1.546600
70,1.538000
80,1.515400
90,1.582800
100,1.419300


TrainOutput(global_step=100, training_loss=1.7809375286102296, metrics={'train_runtime': 199.6605, 'train_samples_per_second': 4.007, 'train_steps_per_second': 0.501, 'total_flos': 1.17845374193664e+16, 'train_loss': 1.7809375286102296})

In [32]:
# Save model
medical_trainer.save_model("/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap13/output/final_model")

In [33]:
# Merge and unload
merged_model = model.merge_and_unload()


## 7º - Pipeline and Chain creation with LangChain

In [34]:
# Defining a pre-prompt string with the instruction
pre_prompt = """[INST]<<SYS>> Analyse the question and answer with the best option. I want a straight answer from the options provided. Do not answer with more then 1 sentence\n"""

In [35]:
# Concatenate pre_prompt with input
prompt = pre_prompt + "Here is my question: {context}" + "[/INST]"

In [36]:
# Creates prompt template
prompt = PromptTemplate(
    input_variables = ["context"],
    template = prompt
)

In [37]:
# Creating the pipeline object
medical_pipe = pipeline(
    task = "text-generation",
    model = merged_model,
    tokenizer = tokenizer,
    max_new_tokens = 512,
    temperature = 0.3,
    top_p = 0.7,
    pad_token_id = tokenizer.eos_token_id,
    use_cache = False,
    do_sample = True
)

Device set to use cuda:0


In [38]:
# Creates Huggingface pipeline
llm_medical_pipe = HuggingFacePipeline(pipeline = medical_pipe)

In [39]:
# Create memory for Chain
memory = ConversationBufferMemory()

In [40]:
# Creates the LLM Chain
medical_chat_llm_chain = LLMChain(
    llm = llm_medical_pipe,
    prompt = prompt,
    verbose = False,
    memory = memory
)

## 8º - Model deployment and usage on the Q&A System

In [41]:
# Prompt
context = '''
###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:
###Options:
A. Adequate supply of safe drinking water
B. Nutrition
C. Provision of free medicines
D. Basic sanitation
'''

In [42]:
# Run the Chain
medical_chat_llm_chain.predict(
    context = context
)

'[INST]<<SYS>> Analyse the question and answer with the best option. I want a straight answer from the options provided. Do not answer with more then 1 sentence\nHere is my question: \n###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:\n###Options:\nA. Adequate supply of safe drinking water\nB. Nutrition\nC. Provision of free medicines\nD. Basic sanitation\n[/INST]  The correct answer is (D) Basic sanitation. According to the Alma Ata declaration, primary health care should include access to basic sanitation, but the other options (A, B, and C) are not explicitly mentioned.'